# Entrenamiento · Cobertura forestal (s4 Jupyter)

Este notebook cierra el flujo de la arquitectura:

```
s3 PostgreSQL  --->  s4 Jupyter (este notebook)  --->  s5 MinIO  --->  s6 Inference API
```

1. Lee la tabla `covertype_raw` que el DAG de Airflow (s2) va acumulando en Postgres.
2. Entrena un `RandomForestClassifier` para predecir `cover_type`.
3. Publica el modelo serializado (joblib) en el bucket `models` de MinIO.

La Inference API (s6) toma automaticamente el objeto mas reciente del bucket.

In [ ]:
# La imagen de Jupyter monta el paquete en /workspace/src (PYTHONPATH).
from cubierta_forestal.infrastructure import postgres_client, minio_client
from cubierta_forestal.application import training, train_pipeline

import pandas as pd

pd.set_option("display.max_columns", None)

## 1. Leer datos desde Postgres (s3)

In [ ]:
df = postgres_client.read_covertype()
print(f"Filas leidas: {len(df)}")
df.head()

In [ ]:
# Distribucion de la variable objetivo (7 clases de cobertura forestal)
df["cover_type"].value_counts().sort_index()

## 2. Entrenar el modelo (s4)

In [ ]:
trained = training.train_model(df)
trained.metrics

In [ ]:
# Ficha del modelo: orden de features y codigos de las columnas categoricas.
trained.metadata()

## 3. Publicar en MinIO (s5)

In [ ]:
from datetime import datetime, timezone
import os

object_name = f"{os.getenv('MODEL_NAME', 'cubierta-forestal')}/covertype-rf-" \
    f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.joblib"

published = minio_client.publish_model(
    trained.pipeline, object_name, metadata=trained.metadata()
)
published

In [ ]:
# Confirmar que el objeto quedo en el bucket (lo que vera la Inference API).
minio_client.list_models()

## Atajo: todo en una sola llamada

`train_pipeline.run()` hace exactamente lo anterior (leer -> entrenar -> publicar)
y devuelve un resumen. Es lo que ejecutaria un job programado.

In [ ]:
train_pipeline.run()